# batchnorm-affine-params — worked example 3: Invert BatchNorm's affine to recover x_hat

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `batchnorm-affine-params`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

BatchNorm's affine map `y = gamma * x_hat + beta` is invertible per channel whenever `gamma != 0`: `x_hat = (y - beta) / gamma`. Recovering the normalized input from the output requires the same per-channel broadcasting reshape, applied in reverse.

## Worked solution

Given the affine output `y` of shape `(B, C, H, W)` and the known per-channel `gamma`, `beta`, we recover the pre-affine normalized tensor `x_hat`.

**Step 1 — algebra.** Start from `y = gamma * x_hat + beta`. Subtract `beta`: `y - beta = gamma * x_hat`. Divide by `gamma`: `x_hat = (y - beta) / gamma`. This holds element-wise within each channel as long as `gamma[c] != 0`.

**Step 2 — reshape params for broadcasting.** Both `gamma` and `beta` are `(C,)`; reshape to `(1, C, 1, 1)` so they line up against the channel axis of `y` and broadcast over `B`, `H`, `W`.

**Step 3 — apply the inverse.** `(y - b) / g`. Subtraction and division both broadcast the `(1, C, 1, 1)` params across the whole tensor.

**Why it works.** A non-zero-slope 1-D affine line is a bijection, so each channel's map is invertible. The inverse is itself affine (slope `1/gamma`, intercept `-beta/gamma`), which is why the same broadcasting machinery applies. We verify by round-tripping: invert, then re-apply the forward affine, and check we get `y` back.

In [ ]:
def bn_affine_invert(y: Tensor, gamma: Tensor, beta: Tensor) -> Tensor:
    g = gamma.view(1, -1, 1, 1)
    b = beta.view(1, -1, 1, 1)
    return (y - b) / g

t.manual_seed(0)
B, C, H, W = 2, 3, 2, 2
x_hat = t.randn(B, C, H, W)
gamma = t.tensor([2.0, 0.5, 4.0])
beta = t.tensor([1.0, -2.0, 0.0])
y = gamma.view(1, -1, 1, 1) * x_hat + beta.view(1, -1, 1, 1)
x_rec = bn_affine_invert(y, gamma, beta)
print(x_rec.shape)
print(t.allclose(x_rec, x_hat, atol=1e-6))